In [4]:
import pandas as pd
import ast
from sklearn.preprocessing import MinMaxScaler

# Load and merge datasets
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')
df = movies.merge(credits, on='title')
df = df[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'vote_count']].dropna()

# Parse JSON-like strings
def extract_names(text, limit=None):
    items = ast.literal_eval(text)
    names = [item['name'] for item in items]
    return names[:limit] if limit else names

def extract_director(text):
    for item in ast.literal_eval(text):
        if item['job'] == 'Director':
            return item['name']
    return ''

# Apply parsing
df['genres']   = df['genres'].apply(lambda x: extract_names(x))
df['keywords'] = df['keywords'].apply(lambda x: extract_names(x, limit=8))
df['cast']     = df['cast'].apply(lambda x: extract_names(x, limit=3))
df['director'] = df['crew'].apply(extract_director)
df['overview'] = df['overview'].str.lower().str.split()

# Create a combined feature signature
def create_signature(row):
    tokens = row['overview'] + row['genres'] + row['keywords'] + row['cast']
    if row['director']:
        tokens += [row['director']] * 2  # weight director
    return tokens

df['signature'] = df.apply(create_signature, axis=1)

# Normalize popularity score
scaler = MinMaxScaler()
v_avg = scaler.fit_transform(df[['vote_average']])
v_cnt = scaler.fit_transform(df[['vote_count']])
df['popularity_score'] = 0.6 * v_avg.flatten() + 0.4 * v_cnt.flatten()

# Jaccard similarity function
def jaccard_similarity(a, b):
    set_a, set_b = set(a), set(b)
    return len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 0

# Recommender function with top_n
def recommend_unique(title, top_n=5):
    title = title.lower()
    if title not in df['title'].str.lower().values:
        print(f"'{title}' not found.")
        return

    idx = df[df['title'].str.lower() == title].index[0]
    base_sig = df.at[idx, 'signature']
    base_genres = set(df.at[idx, 'genres'])

    scores = []
    for i, row in df.iterrows():
        if i == idx:
            continue
        sig_sim = jaccard_similarity(base_sig, row['signature'])
        genre_sim = jaccard_similarity(base_genres, set(row['genres']))
        popularity = row['popularity_score']
        final_score = 0.5 * sig_sim + 0.25 * genre_sim + 0.25 * popularity
        scores.append((i, final_score))

    top_matches = sorted(scores, key=lambda x: x[1], reverse=True)[:top_n]

    print(f"\nTop {top_n} Movies similar to '{df.at[idx, 'title']}':\n")
    for i, _ in top_matches:
        print(df.at[i, 'title'])
#recommend_unique("Avengers: Age of Ultron", top_n=2)
recommend_unique("Pirates of the Caribbean: Dead Man's Chest", top_n=4)


Top 4 Movies similar to 'Pirates of the Caribbean: Dead Man's Chest':

Pirates of the Caribbean: The Curse of the Black Pearl
Pirates of the Caribbean: At World's End
The Lord of the Rings: The Fellowship of the Ring
The Lord of the Rings: The Return of the King
